# The cross-subject generalization gap on CSI-HAR

Same models, same features, same training. The only thing that changes is how the data is
split into train and test:

* **Random split**: stratified 80/20, 3 seeds. Samples from every user appear in both train
  and test, so the model can memorize per-user channel signatures. This is the protocol many
  CSI-HAR papers report.
* **Leave-one-user-out (LOUO)**: 3 folds, train on two users, test on the held-out third.
  No subject appears in both train and test. This is what a deployed device faces on a new
  person.

The difference between the two accuracies is the generalization gap. Attach
**sayakghorai34/csi-har-dataset**; GPU T4, Internet On.


In [1]:
import os, re, json, warnings
from pathlib import Path
import numpy as np, tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")
print("TensorFlow", tf.__version__)
OUT=Path("/kaggle/working"); TARGET_T=64; EPOCHS=60


2026-06-07 21:05:50.705072: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780866350.884503      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780866350.947081      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780866351.416461      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780866351.416501      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780866351.416504      23 computation_placer.cc:177] computation placer alr

TensorFlow 2.19.0


In [2]:
# ---- CSI-HAR loader (identical to the frontier notebook) ----
def find_csi_har_root():
    for c in Path("/kaggle/input").rglob("CSI-HAR-Dataset"):
        if c.is_dir(): return c
    for c in Path("/kaggle/input").rglob("*"):
        if c.is_dir() and (c/"walk").is_dir() and (c/"run").is_dir(): return c
    return None
def load_csi_har_raw(T=TARGET_T):
    root=find_csi_har_root()
    if root is None: raise FileNotFoundError("CSI-HAR-Dataset not attached")
    files=[p for p in root.rglob("*_A.csv") if not p.name.startswith("Annotation")]
    acts=sorted({p.parent.name for p in files}); lmap={a:i for i,a in enumerate(acts)}
    X=[];y=[];users=[]
    for p in files:
        try: a=np.genfromtxt(str(p),delimiter=",")
        except Exception: continue
        if a.ndim==1: a=a.reshape(-1,1)
        if a.shape[0]<2 or a.shape[1]<2: continue
        idx=np.linspace(0,a.shape[0]-1,T).astype(int)
        X.append(a[idx,:].astype(np.float32)); y.append(lmap[p.parent.name])
        m=re.search(r"user_(\d+)_",p.name); users.append(int(m.group(1)) if m else 0)
    X=np.asarray(X,np.float32); y=np.asarray(y,np.int64); users=np.asarray(users)
    m=X.mean(axis=(1,2),keepdims=True); s=X.std(axis=(1,2),keepdims=True)+1e-8; X=((X-m)/s).astype(np.float32)
    print(f"CSI-HAR {X.shape} F={X.shape[2]} classes={acts} users={sorted(set(users.tolist()))}")
    return X,y,users
def csi_features(X):
    mean=X.mean(1); std=X.std(1); mn=X.min(1); mx=X.max(1); rng=mx-mn
    return np.concatenate([mean,std,mn,mx,rng],axis=1).astype(np.float32)


In [3]:
# ---- the two split protocols, each producing a list of (Xtr,ytr,Xte,yte) ----
X,Y,U=load_csi_har_raw(); NCLS=int(Y.max())+1
def folds_random(seeds=3):
    out=[]
    for s in range(seeds):
        sss=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=s)
        tr,te=next(sss.split(X,Y)); out.append((X[tr],Y[tr],X[te],Y[te]))
    return out
def folds_louo():
    out=[]
    for uu in sorted(set(U.tolist())):
        te=U==uu; out.append((X[~te],Y[~te],X[te],Y[te]))
    return out
PROTOCOLS={"random":folds_random(),"louo":folds_louo()}
for k,f in PROTOCOLS.items(): print(k, "folds:", len(f), "test sizes:", [len(t[3]) for t in f])


CSI-HAR (420, 64, 52) F=52 classes=['bend', 'fall', 'lie down', 'run', 'sitdown', 'standup', 'walk'] users=[1, 2, 3]
random folds: 3 test sizes: [84, 84, 84]
louo folds: 3 test sizes: [140, 140, 140]


In [4]:
# ---- models (identical to the frontier notebook) ----
def _factory(name,s):
    if name=="DecisionTree": return DecisionTreeClassifier(max_depth=12,random_state=s)
    if name=="RandomForest": return RandomForestClassifier(n_estimators=20,max_depth=10,random_state=s,n_jobs=-1)
    return MLPClassifier(hidden_layer_sizes=(32,),max_iter=300,random_state=s)
def tiny_cnn(T,F,n,ch):
    i=layers.Input((T,F)); x=layers.Conv1D(ch,7,padding="same",activation="relu")(i)
    x=layers.MaxPool1D(2)(x); x=layers.Conv1D(ch*2,5,padding="same",activation="relu")(x)
    x=layers.GlobalAveragePooling1D()(x); return Model(i,layers.Dense(n)(x),name=f"TinyCNN{ch}")
def tiny_mlp_nn(T,F,n):
    i=layers.Input((T,F)); x=layers.Flatten()(i); x=layers.Dense(32,activation="relu")(x); return Model(i,layers.Dense(n)(x),name="TinyMLP_nn")
def train_nn(builder,Xtr,ytr,Xte,yte,seed=0):
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    m=builder(int(Xtr.shape[1]),int(Xtr.shape[2]),NCLS)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    m.fit(Xtr,ytr,epochs=EPOCHS,batch_size=64,verbose=0)
    return float(accuracy_score(yte,m.predict(Xte,verbose=0).argmax(-1)))
CLASSICAL=["DecisionTree","RandomForest","MLP_stats"]
NEURAL={"TinyCNN16":lambda T,F,n:tiny_cnn(T,F,n,16),"TinyCNN32":lambda T,F,n:tiny_cnn(T,F,n,32),
        "TinyMLP_nn":tiny_mlp_nn}


In [5]:
# ---- run every model under both protocols ----
import pandas as pd
def eval_classical(name,folds):
    accs=[]
    for (Xtr,ytr,Xte,yte) in folds:
        Ftr,Fte=csi_features(Xtr),csi_features(Xte)
        seeds=3 if len(folds)<=3 else 1
        for s in range(seeds):
            nm="MLP" if name=="MLP_stats" else name
            clf=_factory(nm,s) if name!="MLP_stats" else _factory("MLP",s)
            clf.fit(Ftr,ytr); accs.append(accuracy_score(yte,clf.predict(Fte)))
    return float(np.mean(accs)),float(np.std(accs))
def eval_neural(builder,folds):
    accs=[train_nn(builder,*f) for f in folds]
    return float(np.mean(accs)),float(np.std(accs))

rows=[]
for name in CLASSICAL:
    r=eval_classical(name,PROTOCOLS["random"]); l=eval_classical(name,PROTOCOLS["louo"])
    rows.append((name,r[0],r[1],l[0],l[1]))
    print(f"{name:12s} random {r[0]*100:5.1f}  louo {l[0]*100:5.1f}  gap {(r[0]-l[0])*100:5.1f}")
for name,b in NEURAL.items():
    r=eval_neural(b,PROTOCOLS["random"]); l=eval_neural(b,PROTOCOLS["louo"])
    rows.append((name,r[0],r[1],l[0],l[1]))
    print(f"{name:12s} random {r[0]*100:5.1f}  louo {l[0]*100:5.1f}  gap {(r[0]-l[0])*100:5.1f}")

df=pd.DataFrame(rows,columns=["Model","random_mean","random_std","louo_mean","louo_std"])
df["gap_pts"]=(df["random_mean"]-df["louo_mean"])*100
df.to_csv(OUT/"gengap_table.csv",index=False)
json.dump(df.to_dict(orient="records"),open(OUT/"gengap.json","w"),indent=2)
print("\nGENERALIZATION GAP (CSI-HAR)\n"+"-"*52)
print(df.assign(**{c:(df[c]*100).round(1) for c in ["random_mean","louo_mean"]})[["Model","random_mean","louo_mean","gap_pts"]].to_string(index=False))
print(f"\nMean gap across models: {df['gap_pts'].mean():.1f} points")
print("saved gengap_table.csv, gengap.json")
df


DecisionTree random  69.4  louo  40.6  gap  28.9
RandomForest random  83.1  louo  58.1  gap  25.0
MLP_stats    random  85.2  louo  56.9  gap  28.3


I0000 00:00:1780866387.691863      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1780866389.865493     318 service.cc:152] XLA service 0x7fa9d0003940 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780866389.865541     318 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1780866390.105691     318 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780866391.707038     318 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


TinyCNN16    random  92.1  louo  62.1  gap  29.9
TinyCNN32    random  96.4  louo  63.1  gap  33.3
TinyMLP_nn   random  87.7  louo  56.9  gap  30.8

GENERALIZATION GAP (CSI-HAR)
----------------------------------------------------
       Model  random_mean  louo_mean   gap_pts
DecisionTree         69.4       40.6 28.888889
RandomForest         83.1       58.1 24.973545
   MLP_stats         85.2       56.9 28.280423
   TinyCNN16         92.1       62.1 29.920635
   TinyCNN32         96.4       63.1 33.333333
  TinyMLP_nn         87.7       56.9 30.793651

Mean gap across models: 29.4 points
saved gengap_table.csv, gengap.json


,Model,random_mean,random_std,louo_mean,louo_std,gap_pts
0,DecisionTree,0.694444,0.065446,0.405556,0.057811,28.888889
1,RandomForest,0.830688,0.029040,0.580952,0.057735,24.973545
2,MLP_stats,0.851852,0.034187,0.569048,0.019048,28.280423
3,TinyCNN16,0.920635,0.005612,0.621429,0.015430,29.920635
4,TinyCNN32,0.964286,0.019440,0.630952,0.029354,33.333333
5,TinyMLP_nn,0.876984,0.024462,0.569048,0.035154,30.793651


## Reading the result
A large positive gap means the random-split accuracy is inflated by subject leakage and does
not reflect performance on a new person. The honest (LOUO) accuracy is the one a deployed
device achieves. This is the binding constraint on the cheapest WiFi MCU: not memory or
compute (the deployability frontier shows even deep models fit), but cross-subject
generalization.
